# PH-03 Preprocessing - AirGlobal Dataset

Phạm vi xử lý trong notebook này:

- Chỉ đọc một file duy nhất: `data/raw/global_air_quality_2014_2025.csv`
- Không đọc country folders
- Không đọc hoặc tạo datalake
- Xuất dữ liệu đã tiền xử lý vào `outputs/ph03/`
- Giữ nguyên mục tiêu: tạo dataset sạch cho PH-05 Classification

Output chính:

- `outputs/ph03/processed_airglobal_features.pkl`
- `outputs/ph03/processed_airglobal_sample_5000.csv`
- `outputs/ph03/feature_catalog.csv`
- `outputs/ph03/split_summary.csv`
- `outputs/ph03/label_distribution.csv`
- `outputs/ph03/preprocessing_metadata.json`

## 1. Import thư viện

Cell này nạp các thư viện cần thiết cho tiền xử lý dữ liệu.

- `pandas`, `numpy`: xử lý bảng dữ liệu và tính toán số học
- `Path`: xử lý đường dẫn file/folder
- `dataclass`: lưu metadata preprocessing
- `json`: lưu báo cáo metadata

In [1]:
from __future__ import annotations

import json
import re
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

## 2. Cấu hình đường dẫn và tham số chạy

Notebook này chỉ đọc file global CSV. Điều này giúp tránh nhầm lẫn giữa:

- file global tổng hợp
- các file country-level
- các bản copy lồng trong folder khác

PH-03 chỉ tạo processed dataset. Datalake/Data Warehouse sẽ do pha sau thực hiện dựa trên output của PH-03.

In [2]:
def find_project_root() -> Path:
    """Tìm thư mục gốc project.

    Notebook có thể nằm ở root hoặc trong thư mục notebooks/.
    Hàm này đi ngược lên các thư mục cha cho tới khi thấy folder data/.
    """
    current = Path.cwd().resolve()
    for parent in [current] + list(current.parents):
        if (parent / "data").exists():
            return parent
    return current


BASE_DIR = find_project_root()
RAW_ROOT = BASE_DIR / "data" / "raw"
GLOBAL_RAW_CSV = RAW_ROOT / "global_air_quality_2014_2025.csv"

OUTPUT_DIR = BASE_DIR / "outputs" / "ph03"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_COL = "target_aqi_bucket"
DATE_COL = "date"
SPLIT_COL = "split"

# Để tránh file CSV full quá lớn, mặc định chỉ lưu pickle + sample CSV.
SAVE_FULL_CSV = False
SAVE_PARQUET = False

print("Project root:", BASE_DIR)
print("Input global CSV:", GLOBAL_RAW_CSV)
print("Output folder:", OUTPUT_DIR)

Project root: C:\Users\truoan\Documents\DataMining\Final\DATA_FINAL
Input global CSV: C:\Users\truoan\Documents\DataMining\Final\DATA_FINAL\data\raw\global_air_quality_2014_2025.csv
Output folder: C:\Users\truoan\Documents\DataMining\Final\DATA_FINAL\outputs\ph03


## 3. Khai báo schema, target label và các cột xử lý

Cell này định nghĩa:

- cách đổi tên cột về dạng thống nhất
- danh sách pollutant features
- danh sách environmental features có sẵn trong dataset
- các cột có nguy cơ gây data leakage
- các ngưỡng AQI để tạo nhãn nếu thiếu `AQI_Bucket`

In [3]:
COLUMN_RENAME = {
    "Country": "country",
    "State": "state",
    "City": "city",
    "Date": "date",
    "PM2.5 (ug/m3)": "pm25",
    "PM10 (ug/m3)": "pm10",
    "NO (ug/m3)": "no",
    "NO2 (ug/m3)": "no2",
    "NOx (ppb)": "nox",
    "NH3 (ug/m3)": "nh3",
    "CO (mg/m3)": "co",
    "SO2 (ug/m3)": "so2",
    "O3 (ug/m3)": "o3",
    "Benzene (ug/m3)": "benzene",
    "Toluene (ug/m3)": "toluene",
    "Xylene (ug/m3)": "xylene",
    "AQI": "aqi",
    "AQI_Bucket": "aqi_bucket",
    "Wind_Speed (km/h)": "wind_speed",
    "Humidity (%)": "humidity",
    "Deforestation_Rate_%": "deforestation_rate",
    "Industry_Growth_%": "industry_growth",
    "CO2_Emission_MT": "co2_emission",
    "Population_Density_per_SqKm": "population_density",
    "Division": "division",
    "Province": "province",
    "Region": "region",
    "Prefecture": "prefecture",
    "Federal_District": "federal_district",
    "year": "year",
}

POLLUTANT_COLS = [
    "pm25", "pm10", "no", "no2", "nox", "nh3", "co", "so2", "o3",
    "benzene", "toluene", "xylene",
]

ENV_COLS = [
    "wind_speed", "humidity", "deforestation_rate", "industry_growth",
    "co2_emission", "population_density",
]

NUMERIC_BASE_COLS = POLLUTANT_COLS + ENV_COLS

CATEGORICAL_BASE_COLS = [
    "country", "state", "city", "division", "province", "region", "prefecture", "federal_district",
]

# Không đưa các cột này vào feature để tránh dùng đáp án hoặc biến suy ra trực tiếp từ đáp án.
LEAKAGE_COLUMNS = {"aqi", "aqi_bucket", "aqi_bucket_clean", TARGET_COL}

PHYSICAL_LIMITS = {
    "pm25": (0, 1500),
    "pm10": (0, 2500),
    "no": (0, 2000),
    "no2": (0, 2000),
    "nox": (0, 4000),
    "nh3": (0, 2000),
    "co": (0, 100),
    "so2": (0, 2000),
    "o3": (0, 2000),
    "benzene": (0, 500),
    "toluene": (0, 1000),
    "xylene": (0, 1000),
    "aqi": (0, 1000),
    "wind_speed": (0, 300),
    "humidity": (0, 100),
    "deforestation_rate": (0, 100),
    "industry_growth": (-100, 300),
    "co2_emission": (0, 100000),
    "population_density": (0, 200000),
}

AQI_NUMERIC_THRESHOLDS = [50, 100, 200, 300, 400]
AQI_NUMERIC_LABELS = ["Good", "Satisfactory", "Moderate", "Unhealthy", "Very_Unhealthy", "Hazardous"]

LABEL_NORMALIZATION = {
    "excellent": "Good",
    "good": "Good",
    "satisfactory": "Satisfactory",
    "lightly polluted": "Satisfactory",
    "moderate": "Moderate",
    "moderately polluted": "Moderate",
    "unhealthy for sensitive groups": "Unhealthy",
    "unhealthy": "Unhealthy",
    "poor": "Unhealthy",
    "heavily polluted": "Unhealthy",
    "very poor": "Very_Unhealthy",
    "very unhealthy": "Very_Unhealthy",
    "severe": "Hazardous",
    "severely polluted": "Hazardous",
    "hazardous": "Hazardous",
    "unknown": np.nan,
    "nan": np.nan,
    "none": np.nan,
    "": np.nan,
}

LABEL_VI = {
    "Good": "Tốt",
    "Satisfactory": "Khá / Chấp nhận được",
    "Moderate": "Trung bình",
    "Unhealthy": "Kém / Không lành mạnh",
    "Very_Unhealthy": "Rất xấu",
    "Hazardous": "Nguy hại",
}

LAG_PERIODS = [1, 3, 12]
ROLLING_WINDOWS = [3, 12]
ENABLE_ROLLING_FEATURES = False
LAG_BASE_COLS = ["pm25", "pm10", "no2", "o3", "co", "humidity", "wind_speed"]

## 4. Metadata summary

Class này dùng để lưu tóm tắt preprocessing ra file JSON, giúp nhóm dễ ghi báo cáo và debug.

In [4]:
@dataclass
class PreprocessSummary:
    raw_rows_loaded: int
    rows_after_cleaning: int
    duplicate_rows_removed: int
    countries: int
    cities: int
    date_from: str
    date_to: str
    numeric_features: int
    categorical_features: int
    total_features: int
    target_col: str
    label_distribution: Dict[str, int]
    split_distribution: Dict[str, int]
    notes: List[str]


def log(message: str) -> None:
    print(message, flush=True)

## 5. Các hàm chuẩn hóa cột và đọc dữ liệu global CSV

Phần này chỉ đọc một file duy nhất: `data/raw/global_air_quality_2014_2025.csv`.

In [5]:
def normalize_column_name(col: str) -> str:
    col = str(col).strip()
    if col in COLUMN_RENAME:
        return COLUMN_RENAME[col]
    col = re.sub(r"[^0-9a-zA-Z]+", "_", col).strip("_").lower()
    return col


def read_global_raw_dataset() -> pd.DataFrame:
    if not GLOBAL_RAW_CSV.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file global CSV: {GLOBAL_RAW_CSV}. "
            "Hãy đặt file vào data/raw/global_air_quality_2014_2025.csv"
        )

    df = pd.read_csv(GLOBAL_RAW_CSV)
    df["source_file"] = str(GLOBAL_RAW_CSV.relative_to(BASE_DIR))
    log(f"-> Loaded global raw CSV: {len(df):,} rows")
    return df


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalize_column_name(c) for c in df.columns]
    return df

## 6. Tạo target label AQI

Nếu dataset đã có `AQI_Bucket`, notebook sẽ chuẩn hóa nhãn đó.

Nếu thiếu `AQI_Bucket`, notebook tạo nhãn từ cột `AQI` theo các ngưỡng đã khai báo.

Lưu ý: cột `AQI` không được dùng làm feature train model để tránh data leakage.

In [6]:
def normalize_label(value) -> object:
    if pd.isna(value):
        return np.nan
    key = str(value).strip().lower().replace("_", " ")
    return LABEL_NORMALIZATION.get(key, str(value).strip().replace(" ", "_"))


def derive_label_from_aqi(aqi_value) -> object:
    if pd.isna(aqi_value):
        return np.nan
    try:
        val = float(aqi_value)
    except Exception:
        return np.nan

    for threshold, label in zip(AQI_NUMERIC_THRESHOLDS, AQI_NUMERIC_LABELS):
        if val <= threshold:
            return label
    return "Hazardous"


def add_target_label(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "aqi_bucket" in df.columns:
        df["aqi_bucket_clean"] = df["aqi_bucket"].apply(normalize_label)
    else:
        df["aqi_bucket_clean"] = np.nan

    if "aqi" in df.columns:
        derived = df["aqi"].apply(derive_label_from_aqi)
        df[TARGET_COL] = df["aqi_bucket_clean"].fillna(derived)
    else:
        df[TARGET_COL] = df["aqi_bucket_clean"]

    df = df.dropna(subset=[TARGET_COL])
    df = df[df[TARGET_COL].isin(AQI_NUMERIC_LABELS)]
    return df

## 7. Làm sạch dữ liệu cơ bản

Các bước trong cell này:

- chuẩn hóa tên cột
- ép kiểu ngày tháng
- xóa dòng thiếu country/city/date
- chuẩn hóa cột categorical
- ép kiểu numeric
- xóa duplicate
- biến giá trị ngoài giới hạn vật lý thành missing value

In [7]:
def clean_base_data(df: pd.DataFrame) -> Tuple[pd.DataFrame, int, pd.DataFrame]:
    df = standardize_columns(df)

    required = ["country", "city", "date"]
    missing_required = [c for c in required if c not in df.columns]
    if missing_required:
        raise ValueError(f"Dataset thiếu cột bắt buộc: {missing_required}")

    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
    df = df.dropna(subset=[DATE_COL, "country", "city"])

    for col in CATEGORICAL_BASE_COLS:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip().replace({"": pd.NA}).fillna("Unknown")

    for col in set(NUMERIC_BASE_COLS + ["aqi"]):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    before_dups = len(df)
    subset = [c for c in ["country", "state", "city", "date"] if c in df.columns]
    df = df.drop_duplicates(subset=subset, keep="first")
    duplicate_rows_removed = before_dups - len(df)

    invalid_records = []
    for col, (lo, hi) in PHYSICAL_LIMITS.items():
        if col not in df.columns:
            continue
        mask = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
        invalid_count = int(mask.sum())
        if invalid_count > 0:
            invalid_records.append({
                "column": col,
                "invalid_count": invalid_count,
                "min_allowed": lo,
                "max_allowed": hi,
            })
            df.loc[mask, col] = np.nan

    invalid_report = pd.DataFrame(invalid_records)
    return df, duplicate_rows_removed, invalid_report

## 8. Báo cáo missing values, xử lý outlier và imputation

Notebook dùng:

- IQR winsorization: clip outlier theo từng quốc gia thay vì xóa dòng
- hierarchical median imputation: city median -> country median -> global median

In [8]:
def save_missing_report(df: pd.DataFrame, filename: str = "missing_report_before_imputation.csv") -> pd.DataFrame:
    report = []
    for col in df.columns:
        missing = int(df[col].isna().sum())
        report.append({
            "column": col,
            "missing_count": missing,
            "missing_pct": round(missing / max(len(df), 1) * 100, 4),
            "dtype": str(df[col].dtype),
        })
    out = pd.DataFrame(report).sort_values("missing_pct", ascending=False)
    out.to_csv(OUTPUT_DIR / filename, index=False)
    return out


def winsorize_iqr(df: pd.DataFrame, numeric_cols: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    records = []

    for col in numeric_cols:
        if col not in df.columns:
            continue

        total_outliers = 0
        for country, idx in df.groupby("country").groups.items():
            values = df.loc[idx, col].dropna()
            if len(values) < 30:
                continue

            q1 = values.quantile(0.25)
            q3 = values.quantile(0.75)
            iqr = q3 - q1
            if not np.isfinite(iqr) or iqr <= 0:
                continue

            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            mask = (df.loc[idx, col] < lower) | (df.loc[idx, col] > upper)
            count = int(mask.sum())
            if count:
                total_outliers += count
                df.loc[idx, col] = df.loc[idx, col].clip(lower=lower, upper=upper)

        records.append({"column": col, "outlier_count_clipped": total_outliers})

    report = pd.DataFrame(records).sort_values("outlier_count_clipped", ascending=False)
    return df, report


def impute_numeric_by_hierarchy(df: pd.DataFrame, numeric_cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    for col in numeric_cols:
        if col not in df.columns:
            continue

        city_median = df.groupby(["country", "city"])[col].transform("median")
        country_median = df.groupby("country")[col].transform("median")
        global_median = df[col].median()
        if pd.isna(global_median):
            global_median = 0.0

        df[col] = df[col].fillna(city_median).fillna(country_median).fillna(global_median)

    return df

## 9. Feature engineering

Các feature được tạo thêm gồm:

- time features: year, month, quarter, season, sin/cos chu kỳ
- pollutant ratios: ví dụ PM2.5/PM10, NO2/NOx
- environmental interactions: PM2.5 với humidity/wind speed
- frequency encoding cho country/city/state
- lag/rolling features theo chuỗi thời gian của từng country-city

In [9]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["year"] = df[DATE_COL].dt.year
    df["month"] = df[DATE_COL].dt.month
    df["quarter"] = df[DATE_COL].dt.quarter
    df["day_of_year"] = df[DATE_COL].dt.dayofyear
    df["decade"] = (df["year"] // 10) * 10
    df["is_covid_period"] = df["year"].isin([2020, 2021]).astype(int)

    def season_from_month(month: int) -> str:
        if month in [12, 1, 2]:
            return "Winter"
        if month in [3, 4, 5]:
            return "Spring"
        if month in [6, 7, 8]:
            return "Summer"
        return "Autumn"

    df["season"] = df["month"].apply(season_from_month)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    return df


def safe_div(numerator: pd.Series, denominator: pd.Series, eps: float = 1e-6) -> pd.Series:
    return numerator / (denominator.replace(0, np.nan) + eps)


def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if {"pm25", "pm10"}.issubset(df.columns):
        df["pm25_pm10_ratio"] = safe_div(df["pm25"], df["pm10"])
        df["pm10_minus_pm25"] = df["pm10"] - df["pm25"]

    if {"no2", "nox"}.issubset(df.columns):
        df["no2_nox_ratio"] = safe_div(df["no2"], df["nox"])

    if {"benzene", "toluene"}.issubset(df.columns):
        df["benzene_toluene_ratio"] = safe_div(df["benzene"], df["toluene"])

    if {"so2", "no2"}.issubset(df.columns):
        df["so2_no2_ratio"] = safe_div(df["so2"], df["no2"])

    if {"pm25", "humidity"}.issubset(df.columns):
        df["pm25_humidity_interaction"] = df["pm25"] * df["humidity"]
        df["high_humidity_flag"] = (df["humidity"] >= 80).astype(int)

    if {"pm25", "wind_speed"}.issubset(df.columns):
        df["pm25_wind_dispersion"] = df["pm25"] / (df["wind_speed"] + 1)
        df["low_wind_flag"] = (df["wind_speed"] < 5).astype(int)

    if {"population_density", "industry_growth"}.issubset(df.columns):
        df["industry_density_interaction"] = df["population_density"] * df["industry_growth"]

    if {"co2_emission", "population_density"}.issubset(df.columns):
        df["co2_per_density"] = df["co2_emission"] / (df["population_density"] + 1)

    for col in ["country", "city", "state"]:
        if col in df.columns:
            freq = df[col].value_counts(normalize=True)
            df[f"{col}_freq"] = df[col].map(freq).fillna(0)

    if {"country", "city"}.issubset(df.columns):
        city_key = df["country"].astype(str) + "__" + df["city"].astype(str)
        freq = city_key.value_counts(normalize=True)
        df["country_city_freq"] = city_key.map(freq).fillna(0)

    return df


def add_lag_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(["country", "city", DATE_COL])
    group_cols = ["country", "city"]
    valid_cols = [c for c in LAG_BASE_COLS if c in df.columns]

    for col in valid_cols:
        grouped_col = df.groupby(group_cols, sort=False)[col]

        for lag in LAG_PERIODS:
            new_col = f"{col}_lag_{lag}"
            df[new_col] = grouped_col.shift(lag)
            df[new_col] = df[new_col].fillna(df[col])

        if ENABLE_ROLLING_FEATURES:
            for window in ROLLING_WINDOWS:
                new_col = f"{col}_roll_mean_{window}"
                df[new_col] = grouped_col.transform(
                    lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
                )
                df[new_col] = df[new_col].fillna(df[col])

    return df

## 10. Chia train/validation/test theo thời gian

Cách chia theo thời gian giúp giảm data leakage so với chia ngẫu nhiên, vì model được train trên dữ liệu cũ hơn và test trên dữ liệu mới hơn.

In [10]:
def split_temporally(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(DATE_COL)
    unique_dates = np.array(sorted(df[DATE_COL].dropna().unique()))

    if len(unique_dates) < 5:
        rng = np.random.default_rng(RANDOM_STATE)
        rand = rng.random(len(df))
        df[SPLIT_COL] = np.where(rand < 0.7, "train", np.where(rand < 0.85, "val", "test"))
        return df

    train_cut = unique_dates[int(len(unique_dates) * 0.70)]
    val_cut = unique_dates[int(len(unique_dates) * 0.85)]

    df[SPLIT_COL] = "test"
    df.loc[df[DATE_COL] < train_cut, SPLIT_COL] = "train"
    df.loc[(df[DATE_COL] >= train_cut) & (df[DATE_COL] < val_cut), SPLIT_COL] = "val"

    if df.loc[df[SPLIT_COL] == "train", TARGET_COL].nunique() < 2:
        log("Warning: Temporal split produced too few train classes. Fallback to random split.")
        rng = np.random.default_rng(RANDOM_STATE)
        rand = rng.random(len(df))
        df[SPLIT_COL] = np.where(rand < 0.7, "train", np.where(rand < 0.85, "val", "test"))

    return df

## 11. Tạo feature catalog

Feature catalog giúp PH-05 biết cột nào là numeric, cột nào là categorical, và mỗi feature thuộc nhóm nào.

In [11]:
def build_feature_catalog(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str], List[str]]:
    excluded = set([DATE_COL, SPLIT_COL, TARGET_COL, "aqi", "aqi_bucket", "aqi_bucket_clean", "source_file"])
    feature_cols = [c for c in df.columns if c not in excluded]

    categorical_features = [
        c for c in feature_cols
        if str(df[c].dtype) in ["object", "string", "category"]
    ]
    numeric_features = [c for c in feature_cols if c not in categorical_features]

    catalog_records = []
    for col in feature_cols:
        if col in categorical_features:
            group = "categorical/geography"
        elif any(token in col for token in ["lag", "roll"]):
            group = "lag_rolling"
        elif col in ["year", "month", "quarter", "day_of_year", "decade", "month_sin", "month_cos", "doy_sin", "doy_cos", "is_covid_period"]:
            group = "time"
        elif any(token in col for token in ["ratio", "interaction", "flag", "dispersion", "freq", "density"]):
            group = "domain_engineered"
        else:
            group = "base_numeric"

        catalog_records.append({
            "feature": col,
            "dtype": str(df[col].dtype),
            "feature_type": "categorical" if col in categorical_features else "numeric",
            "feature_group": group,
            "missing_count": int(df[col].isna().sum()),
        })

    catalog = pd.DataFrame(catalog_records)
    return catalog, numeric_features, categorical_features

## 12. Chạy pipeline preprocessing

Cell này chạy toàn bộ PH-03 từ đọc dữ liệu global CSV tới lưu output.

In [12]:
print("=" * 80)
print("PH-03 PREPROCESSING")
print("=" * 80)

notes: List[str] = []

log("1) Loading global dataset...")
raw_df = read_global_raw_dataset()
raw_rows = len(raw_df)

log("\n2) Cleaning schema, dates, duplicates and invalid physical values...")
df, duplicate_removed, invalid_report = clean_base_data(raw_df)
invalid_report.to_csv(OUTPUT_DIR / "physical_invalid_values_report.csv", index=False)
log(f"-> After basic cleaning: {len(df):,} rows | duplicates removed: {duplicate_removed:,}")

log("\n3) Creating target label and avoiding AQI leakage...")
df = add_target_label(df)
log(f"-> Rows with valid target: {len(df):,} | classes: {df[TARGET_COL].nunique()}")

log("\n4) Missing pattern report before imputation...")
missing_before = int(df.isna().sum().sum())
missing_report = save_missing_report(df)
log(f"-> Missing cells before imputation: {missing_before:,}")

existing_numeric_cols = [c for c in NUMERIC_BASE_COLS if c in df.columns]

log("\n5) IQR outlier treatment and hierarchical median imputation...")
df, outlier_report = winsorize_iqr(df, existing_numeric_cols)
outlier_report.to_csv(OUTPUT_DIR / "outlier_iqr_report.csv", index=False)
df = impute_numeric_by_hierarchy(df, existing_numeric_cols)
missing_after_base = int(df[existing_numeric_cols].isna().sum().sum()) if existing_numeric_cols else 0
log(f"-> Missing numeric base cells after imputation: {missing_after_base:,}")

log("\n6) Feature engineering: time, ratios, interactions, lag/rolling...")
df = add_time_features(df)
df = add_domain_features(df)
df = add_lag_rolling_features(df)

numeric_cols_now = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
df[numeric_cols_now] = df[numeric_cols_now].replace([np.inf, -np.inf], np.nan)
numeric_medians = df[numeric_cols_now].median(numeric_only=True)
df[numeric_cols_now] = df[numeric_cols_now].fillna(numeric_medians).fillna(0)

for col in CATEGORICAL_BASE_COLS + ["season"]:
    if col in df.columns:
        df[col] = df[col].astype("string").fillna("Unknown")

log("\n7) Temporal train/validation/test split...")
df = split_temporally(df)

catalog, numeric_features, categorical_features = build_feature_catalog(df)
catalog.to_csv(OUTPUT_DIR / "feature_catalog.csv", index=False)

label_dist = df[TARGET_COL].value_counts().reindex(AQI_NUMERIC_LABELS).fillna(0).astype(int)
label_dist_df = pd.DataFrame({
    "label": label_dist.index,
    "label_vi": [LABEL_VI.get(x, x) for x in label_dist.index],
    "rows": label_dist.values,
    "pct": np.round(label_dist.values / max(label_dist.values.sum(), 1) * 100, 4),
})
label_dist_df.to_csv(OUTPUT_DIR / "label_distribution.csv", index=False)

split_summary = df[SPLIT_COL].value_counts().reindex(["train", "val", "test"]).fillna(0).astype(int)
split_summary_df = pd.DataFrame({"split": split_summary.index, "rows": split_summary.values})
split_summary_df.to_csv(OUTPUT_DIR / "split_summary.csv", index=False)

log(f"-> Split: {split_summary_df.to_dict(orient='records')}")
log(f"-> Features: {len(numeric_features)} numeric + {len(categorical_features)} categorical = {len(numeric_features) + len(categorical_features)}")

PH-03 PREPROCESSING
1) Loading global dataset...
-> Loaded global raw CSV: 331,920 rows

2) Cleaning schema, dates, duplicates and invalid physical values...
-> After basic cleaning: 331,920 rows | duplicates removed: 0

3) Creating target label and avoiding AQI leakage...
-> Rows with valid target: 331,920 | classes: 6

4) Missing pattern report before imputation...
-> Missing cells before imputation: 825

5) IQR outlier treatment and hierarchical median imputation...
-> Missing numeric base cells after imputation: 0

6) Feature engineering: time, ratios, interactions, lag/rolling...

7) Temporal train/validation/test split...
-> Split: [{'split': 'train', 'rows': 230500}, {'split': 'val', 'rows': 50710}, {'split': 'test', 'rows': 50710}]
-> Features: 64 numeric + 4 categorical = 68


## 13. Lưu processed dataset và metadata

Notebook mặc định lưu full dataset ở dạng `.pkl` vì nhanh hơn CSV và giữ datatype tốt hơn.

Nếu thật sự cần CSV full, đổi `SAVE_FULL_CSV = True` ở section cấu hình.

In [13]:
log("8) Saving processed dataset and metadata...")

# Chuyển date sang ISO string để output dễ đọc và portable hơn.
df[DATE_COL] = pd.to_datetime(df[DATE_COL]).dt.strftime("%Y-%m-%d")

processed_pkl = OUTPUT_DIR / "processed_airglobal_features.pkl"
df.to_pickle(processed_pkl)

df.head(5000).to_csv(OUTPUT_DIR / "processed_airglobal_sample_5000.csv", index=False)

if SAVE_FULL_CSV:
    processed_csv = OUTPUT_DIR / "processed_airglobal_features.csv"
    df.to_csv(processed_csv, index=False)
    notes.append("Saved full CSV because SAVE_FULL_CSV=True.")
else:
    notes.append("Full CSV skipped by default for speed. Set SAVE_FULL_CSV=True if needed.")

if SAVE_PARQUET:
    try:
        df.to_parquet(OUTPUT_DIR / "processed_airglobal_features.parquet", index=False)
        notes.append("Saved parquet successfully.")
    except Exception as exc:
        notes.append(f"Parquet export skipped: {type(exc).__name__}.")
        log("Warning: Parquet export skipped. Install pyarrow or fastparquet if parquet is needed.")

date_from = str(pd.to_datetime(df[DATE_COL]).min().date()) if not df.empty else ""
date_to = str(pd.to_datetime(df[DATE_COL]).max().date()) if not df.empty else ""

summary = PreprocessSummary(
    raw_rows_loaded=int(raw_rows),
    rows_after_cleaning=int(len(df)),
    duplicate_rows_removed=int(duplicate_removed),
    countries=int(df["country"].nunique()) if "country" in df.columns else 0,
    cities=int(df["city"].nunique()) if "city" in df.columns else 0,
    date_from=date_from,
    date_to=date_to,
    numeric_features=int(len(numeric_features)),
    categorical_features=int(len(categorical_features)),
    total_features=int(len(numeric_features) + len(categorical_features)),
    target_col=TARGET_COL,
    label_distribution={str(k): int(v) for k, v in label_dist.to_dict().items()},
    split_distribution={str(k): int(v) for k, v in split_summary.to_dict().items()},
    notes=notes,
)

with open(OUTPUT_DIR / "preprocessing_metadata.json", "w", encoding="utf-8") as f:
    json.dump(asdict(summary), f, indent=2, ensure_ascii=False)

log(f"-> Saved: {processed_pkl.relative_to(BASE_DIR)}")
log("-> Saved reports: missing_report_before_imputation.csv, outlier_iqr_report.csv, feature_catalog.csv")
log("-> PH-03 DONE")
log(f"Output folder: {OUTPUT_DIR.relative_to(BASE_DIR)}")
log("Next: python src/train_classification.py")

8) Saving processed dataset and metadata...
-> Saved: outputs\ph03\processed_airglobal_features.pkl
-> Saved reports: missing_report_before_imputation.csv, outlier_iqr_report.csv, feature_catalog.csv
-> PH-03 DONE
Output folder: outputs\ph03
Next: python src/train_classification.py


## 14. Kiểm tra nhanh output

Cell này dùng để kiểm tra shape, phân phối split và phân phối target sau khi chạy xong.

In [14]:
print("Processed shape:", df.shape)
print("\nSplit distribution:")
print(df[SPLIT_COL].value_counts())
print("\nTarget distribution:")
print(df[TARGET_COL].value_counts())
print("\nOutput files:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print("-", path.relative_to(BASE_DIR))

Processed shape: (331920, 75)

Split distribution:
split
train    230500
val       50710
test      50710
Name: count, dtype: int64

Target distribution:
target_aqi_bucket
Moderate          124098
Good               89333
Unhealthy          54498
Satisfactory       53195
Very_Unhealthy      8045
Hazardous           2751
Name: count, dtype: int64

Output files:
- outputs\ph03\feature_catalog.csv
- outputs\ph03\label_distribution.csv
- outputs\ph03\missing_report_before_imputation.csv
- outputs\ph03\outlier_iqr_report.csv
- outputs\ph03\physical_invalid_values_report.csv
- outputs\ph03\preprocessing_metadata.json
- outputs\ph03\processed_airglobal_features.pkl
- outputs\ph03\processed_airglobal_sample_5000.csv
- outputs\ph03\split_summary.csv
